In [139]:
import os
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from case_study.utils import set_matplot_style, BI_PART_COLORS, process_metadata

from src.utils import load_env, get_logger, load_json, save_to_json
from src.experiment_config import ExperimentConfig
from src.data_loading import DatasetLoader

RANDOM_SEED = 42
N_SAMPLES = 50

set_matplot_style()
env_vars = load_env()
logger = get_logger("case study")

In [140]:
config = ExperimentConfig(
    task_type="analysis",
    task_name="podcast_case_study",
    dataset="podcasts",
    logger=logger,
    env_vars=env_vars,
    skip_load=False
)
data_loader = DatasetLoader(config)

In [141]:
mets_df = data_loader.load_metaphor_classification_results(metaphors_only=True)
mets_df["doc_id"] = mets_df["doc_id"].apply(lambda x: "-".join((x.split("_")[0]).split("-")[:-1]))
mets_df["met_id"] = mets_df["id"]
mets_df = mets_df[["met_id", "doc_id", "target_word"]]

2026-07-29 19:10:25,545 - case study - INFO - Loading qwen 'binary metaphor' (specified best model) classification results...
2026-07-29 19:10:25,545 - case study - INFO - Loading qwen 'binary metaphor' (specified best model) classification results...
2026-07-29 19:10:25,545 - case study - INFO - Loading qwen 'binary metaphor' (specified best model) classification results...
2026-07-29 19:10:25,545 - case study - INFO - Loading qwen 'binary metaphor' (specified best model) classification results...
2026-07-29 19:10:25,545 - case study - INFO - Loading qwen 'binary metaphor' (specified best model) classification results...
2026-07-29 19:10:25,545 - case study - INFO - Loading qwen 'binary metaphor' (specified best model) classification results...
2026-07-29 19:10:25,545 - case study - INFO - Loading qwen 'binary metaphor' (specified best model) classification results...
2026-07-29 19:10:25,545 - case study - INFO - Loading qwen 'binary metaphor' (specified best model) classification res

In [142]:
# read in channel data, join
pod_df = data_loader.load_raw_data()
pod_df["doc_id"] = pod_df["id"]
pod_df = pod_df[["doc_id", "polarity"]]
confirmed_mets_df = mets_df.merge(pod_df, on="doc_id", how="left")

In [143]:
EXCLUDED_WORDS = ["you", "i", "it", "this", "that"]
def sample_doc(group: pd.DataFrame, n: int = N_SAMPLES, seed: int = RANDOM_SEED) -> pd.DataFrame:
    lower_target = group["target_word"].str.lower()
    is_excluded = lower_target.isin(EXCLUDED_WORDS)

    preferred = group[~is_excluded]
    excluded = group[is_excluded]

    if len(preferred) >= n:
        result = preferred.sample(n=n, random_state=seed)
    else:
        remaining = n - len(preferred)
        excluded_sample = excluded.sample(n=min(remaining, len(excluded)), random_state=seed)
        result = pd.concat([preferred, excluded_sample], axis=0)

    result["doc_id"] = group.name  # ensure doc_id survives regardless of pandas version behavior
    return result


def build_sampled_df(confirmed_mets_df: pd.DataFrame) -> pd.DataFrame:
    # Count rows per doc and drop docs with fewer than 50
    doc_counts = confirmed_mets_df.groupby("doc_id").size()
    valid_docs = doc_counts[doc_counts >= N_SAMPLES].index

    filtered_df = confirmed_mets_df[confirmed_mets_df["doc_id"].isin(valid_docs)]

    sampled_df = (
        filtered_df.groupby("doc_id", group_keys=False)
        .apply(lambda g: sample_doc(g))
        .reset_index(drop=True)
    )

    return sampled_df


# Example usage:
sampled_df = build_sampled_df(confirmed_mets_df)

print(len(sampled_df["met_id"].unique()))
print(sampled_df.groupby("doc_id").size())
print(sampled_df.groupby("polarity").size())

4650
doc_id
07ccd0c9-c020-446c-95ff-766a07bfe262           50
0ba7df2a-7b29-11ed-8350-6f9d2dcb081f           50
0bbc13d2-7b29-11ed-8350-bf224b5f8c09           50
0e5efcaa-8b85-11ed-95fd-3fc9ab0ea880           50
10672768-1932-4440-9494-b3cd0026c419           50
                                               ..
f2b794b4-ecee-4a95-8346-91a106142e47           50
f5967886-d9e9-11f0-b2b6-432c4e914b8e           50
f74de610-99d5-11ee-996f-3fece011ee96           50
f7d6aefd-4cf0-4acf-a769-941184e0b594           50
tag:audioboom.com,2026-02-06:/posts/8857998    50
Length: 93, dtype: int64
polarity
left     2250
right    2400
dtype: int64


In [144]:
# now load classification results and sample
full_mc_path = f"{env_vars['RESULTS_DIR']}/metaphor_classification/llm/podcasts_source_verb_target_noun_binary_met_class_qwen-complete.json"
full_mc_w_config = load_json(full_mc_path)
full_mc_data = full_mc_w_config["data"]

In [145]:
sampled_data = {}
for s_met_id in list(sampled_df["met_id"].to_list()):
    sampled_data[s_met_id] = full_mc_data[s_met_id]
print(len(sampled_data)) 

4650


In [146]:
# save sampled to json
save_path = f"{env_vars['RESULTS_DIR']}/metaphor_classification/llm"
save_filename = "podcasts_source_verb_target_noun_binary_met_class_qwen.json"
full_mc_w_config["data"] = sampled_data
save_to_json(full_mc_w_config, save_path, save_filename)

In [147]:
# now shorten target noun predictions
full_tnp_path = f"{env_vars['RESULTS_DIR']}/feature_extractor/llm/podcasts_target_ppto_class_qwen-complete.json"
full_tnp_w_config = load_json(full_tnp_path)
full_tnp_data = full_tnp_w_config["data"]

In [148]:
sampled_tn_data = {}
for s_met_id in list(sampled_df["met_id"].to_list()):
    sampled_tn_data[s_met_id] = full_tnp_data[s_met_id]
print(len(sampled_tn_data)) 

4650


In [149]:
# save sampled to json
save_path = f"{env_vars['RESULTS_DIR']}/feature_extractor/llm"
save_filename = "podcasts_target_ppto_class_qwen.json"
full_tnp_w_config["data"] = sampled_tn_data
save_to_json(full_tnp_w_config, save_path, save_filename)